# SIS #2 — Pastebin Protocol Implementation (Python)

**Steps 3–5: implementation, reliability, and benchmarks** · gRPC/Protobuf vs REST/JSON.

Run all cells to execute the benchmarks; the last cell stops the servers. The benchmark results are saved as CSV in `sis2_pastebin_runtime/`.

**Assignment caveat:** the original SIS #2 specifies Java. This notebook is a Python prototype and requires instructor approval if submitted in place of Java.

## 1. Install dependencies
Run once, then restart the kernel if Jupyter asks.

In [ ]:
# %pip install -q "grpcio>=1.60" "grpcio-tools>=1.60" "protobuf>=4.25" "fastapi>=0.110" "uvicorn>=0.27" "httpx>=0.27" "pydantic>=2"


### Разбор: зависимости и окружение

**Роль блока.** Подготавливает библиотеки для всего эксперимента. `%pip` — команда IPython/Jupyter, которая устанавливает пакеты в окружение текущего ядра. `-q` уменьшает объём вывода, `>=` задаёт нижнюю границу версии, но не фиксирует конкретный набор версий.

`grpcio` реализует клиент и сервер gRPC; `grpcio-tools` компилирует контракт; `protobuf` обслуживает сообщения. FastAPI и Uvicorn нужны для REST, HTTPX — для HTTP-клиента, Pydantic — для модели запроса. Для новых графиков дополнительно нужен `matplotlib`; если его нет, установи отдельной командой `%pip install matplotlib`. `pandas` в этом ноутбуке необязателен.

**Текущее состояние:** строка начинается с `#`, поэтому ячейка ничего не устанавливает. Это нормально для уже подготовленного окружения. При отсутствии библиотек нужно убрать `#` и выполнить установку; повторная установка не запускает серверы.

**Вопрос преподавателя:** «Чем `%pip` отличается от запуска `pip` в случайном терминале?» — Он относится к окружению выбранного ядра; терминальный `pip` может установить библиотеку в другой Python.

**Вопрос:** «Можно ли воспроизвести точные версии по этим строкам?» — Нет: `>=` допускает новые версии. Для строгого воспроизведения фиксируют версии и окружение эксперимента.


## 2. Configuration and Protocol Buffers contract
The `.proto` file is generated on disk, then compiled to Python stubs. The gRPC request/response fields match the REST API.

In [ ]:
from pathlib import Path
import sys, subprocess, uuid, time, json, sqlite3, secrets, string, threading, socket
from datetime import datetime, timezone

WORKDIR = Path.cwd() / "sis2_pastebin_runtime"
WORKDIR.mkdir(exist_ok=True)
DB_PATH = WORKDIR / "pastebin.sqlite3"
GRPC_PORT, REST_PORT = 50071, 18080
MAX_BYTES = 1_048_576
MIN_TTL, MAX_TTL, DEFAULT_TTL = 60, 30 * 24 * 3600, 7 * 24 * 3600
BASE_URL = f"http://127.0.0.1:{REST_PORT}"

PROTO = r'''syntax = "proto3";
package pastebin;
service Pastebin {
  rpc CreateSnippet(CreateSnippetRequest) returns (CreateSnippetResponse);
  rpc GetSnippet(GetSnippetRequest) returns (GetSnippetResponse);
}
message CreateSnippetRequest {
  string content = 1;
  string title = 2;
  string language = 3;
  int32 ttl_seconds = 4;
  string idempotency_key = 5;
}
message CreateSnippetResponse {
  string id = 1;
  string url = 2;
  string created_at = 3;
  string expires_at = 4;
}
message GetSnippetRequest { string id = 1; }
message GetSnippetResponse {
  string id = 1;
  string content = 2;
  string title = 3;
  string language = 4;
  string created_at = 5;
  string expires_at = 6;
}
'''
(WORKDIR / "pastebin.proto").write_text(PROTO, encoding="utf-8")
subprocess.run([sys.executable, "-m", "grpc_tools.protoc", f"-I{WORKDIR}", f"--python_out={WORKDIR}", f"--grpc_python_out={WORKDIR}", str(WORKDIR / "pastebin.proto")], check=True)
sys.path.insert(0, str(WORKDIR))
import pastebin_pb2 as pb
import pastebin_pb2_grpc as pb_grpc
print("Protocol Buffers stubs generated in", WORKDIR)

### Разбор блока 2: конфигурация и протокол

**Задача блока:** подготовить общие настройки и описать контракт gRPC — какие операции доступны, что клиент передаёт и что получает в ответ. Здесь серверы ещё не запускаются.

**1. Импорты и рабочие файлы.** `Path` нужен для путей; `sys` — для выбора Python и настройки импорта; `subprocess` — для запуска компилятора протокола. Остальные модули понадобятся дальше: `uuid` для ключей идемпотентности, `time` для времени, `json` для сериализации, `sqlite3` для базы, `secrets` и `string` для случайных ID, `threading` для потоков. `datetime` и `timezone` помогают представить время в UTC. `socket` импортирован, но далее не используется.

`Path.cwd()` возвращает текущую рабочую папку ядра, которая не обязательно совпадает с папкой самого ноутбука. В ней создаётся `sis2_pastebin_runtime`; `exist_ok=True` разрешает использовать уже существующую папку. `DB_PATH` задаёт путь к базе SQLite; саму базу откроет следующий блок.

**2. Настройки.** `GRPC_PORT = 50071` и `REST_PORT = 18080` разделяют два сервера по портам. `MAX_BYTES = 1_048_576` — ровно 1 МиБ; подчёркивания в числе нужны только для читаемости. TTL (*time to live*) — срок жизни записи: минимум 60 секунд, максимум 30 дней, по умолчанию 7 дней. Выражения с `24 * 3600` переводят дни в секунды. `BASE_URL` — адрес REST-сервера на текущем компьютере; `f"...{REST_PORT}"` подставляет порт в строку.

**3. Строка `PROTO`.** Это текст на языке Protocol Buffers, а не Python-код. Тройные кавычки позволяют записать несколько строк; префикс `r` отключает обычную обработку обратных слешей в Python-строке. `syntax = "proto3"` задаёт версию синтаксиса, `package pastebin` — пространство имён контракта.

`service Pastebin` объявляет две операции удалённого вызова (RPC): `CreateSnippet` создаёт запись, `GetSnippet` читает её. В скобках указано сообщение запроса, после `returns` — сообщение ответа. `message` описывает структуру данных, `string` означает строку, `int32` — знаковое 32-битное целое. Числа после `=` — номера полей для двоичного кодирования, **не значения по умолчанию**.

| Сообщение | Что содержит |
|---|---|
| `CreateSnippetRequest` | Текст `content`, заголовок `title`, язык `language`, TTL `ttl_seconds` и ключ повтора `idempotency_key`. |
| `CreateSnippetResponse` | ID, публичный URL и время создания/истечения срока. Сам текст здесь не возвращается. |
| `GetSnippetRequest` | Только ID нужной записи. |
| `GetSnippetResponse` | ID, текст, заголовок, язык и обе временные метки. |

У обычных скалярных полей `proto3` при отсутствии значения строка читается как `''`, число — как `0`. В блоке gRPC эти значения частично заменяются настройками по умолчанию. Сам контракт не проверяет предел 1 МиБ или допустимый TTL: это делает общий сервис.

**4. Генерация Python-модулей.** `write_text(..., encoding="utf-8")` сохраняет контракт в `pastebin.proto`. `subprocess.run(...)` запускает `grpc_tools.protoc` через тот же Python, в котором работает ядро (`sys.executable`). Ключ `-I` задаёт каталог поиска `.proto`, `--python_out` — каталог классов сообщений, `--grpc_python_out` — каталог кода gRPC. `check=True` прерывает выполнение с исключением, если компилятор завершится с ошибкой.

Получаются два модуля: `pastebin_pb2` содержит сообщения и их сериализацию, `pastebin_pb2_grpc` — клиентский интерфейс и основу серверного обработчика. `sys.path.insert(0, ...)` позволяет импортировать их из рабочей папки; `as pb` и `as pb_grpc` задают короткие имена. Для изменения контракта нужно править `PROTO`, затем заново генерировать модули.


**Главное для защиты:** RPC `CreateSnippet` / `GetSnippet`, номера полей в `PROTO`, запуск `grpc_tools.protoc` и назначение `pb` / `pb_grpc`.

**Вопрос преподавателя:** «Что означает `string content = 1`?» — Строковое поле с номером 1 в бинарном контракте; число не является значением по умолчанию. Номера используются при кодировании; нельзя произвольно менять их смысл для уже существующих клиентов.

**Вопрос:** «Почему нужны два сгенерированных модуля?» — Один содержит классы сообщений, другой — клиентский stub, серверную основу и регистрацию методов. Генерация создаёт транспортную обвязку, но не реализует хранение.


## 3. SQLite storage and shared business logic
SQLite transactions and unique constraints protect duplicate creation. For repeated idempotency keys, request fingerprints must match. Expired snippets are never returned even before cleanup.

In [ ]:
import hashlib

class ServiceError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status, self.message = status, message

class PastebinService:
    def __init__(self, db_path):
        self.db_path = str(db_path)
        self._lock = threading.RLock()
        with self._connect() as db:
            db.execute("PRAGMA journal_mode=WAL")
            db.execute("""CREATE TABLE IF NOT EXISTS snippets (
                id TEXT PRIMARY KEY, content TEXT NOT NULL, title TEXT,
                language TEXT NOT NULL, size_bytes INTEGER NOT NULL,
                created_at REAL NOT NULL, expires_at REAL NOT NULL)""")
            db.execute("CREATE INDEX IF NOT EXISTS snippets_expires_idx ON snippets(expires_at)")
            db.execute("""CREATE TABLE IF NOT EXISTS idempotency_keys (
                key TEXT PRIMARY KEY, fingerprint TEXT NOT NULL,
                response_json TEXT NOT NULL, created_at REAL NOT NULL)""")

    def _connect(self):
        db = sqlite3.connect(self.db_path, timeout=5, check_same_thread=False)
        db.row_factory = sqlite3.Row
        return db

    @staticmethod
    def _iso(ts):
        return datetime.fromtimestamp(ts, timezone.utc).isoformat().replace('+00:00', 'Z')

    def create(self, content, title='', language='plaintext', ttl_seconds=DEFAULT_TTL, key=None):
        if not isinstance(content, str):
            raise ServiceError(422, 'content must be a string')
        size = len(content.encode('utf-8'))
        if size > MAX_BYTES:
            raise ServiceError(413, 'content exceeds 1 MiB')
        if size < 1:
            raise ServiceError(422, 'content must not be empty')
        if not isinstance(ttl_seconds, int) or not MIN_TTL <= ttl_seconds <= MAX_TTL:
            raise ServiceError(422, 'ttl_seconds out of range')
        if not isinstance(title, str) or len(title) > 200:
            raise ServiceError(422, 'invalid title')
        if not isinstance(language, str) or not 1 <= len(language) <= 32:
            raise ServiceError(422, 'invalid language')
        if key is not None:
            try: key = str(uuid.UUID(key))
            except (ValueError, TypeError, AttributeError):
                raise ServiceError(422, 'idempotency key must be a UUID')
        payload = dict(content=content, title=title, language=language, ttl_seconds=ttl_seconds)
        fingerprint = hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        with self._lock, self._connect() as db:
            db.execute('BEGIN IMMEDIATE')
            if key:
                old = db.execute('SELECT fingerprint,response_json FROM idempotency_keys WHERE key=?', (key,)).fetchone()
                if old:
                    if old['fingerprint'] != fingerprint:
                        raise ServiceError(409, 'idempotency key reused with different content')
                    return json.loads(old['response_json'])
            now = time.time()
            for _ in range(3):
                snippet_id = ''.join(secrets.choice(string.ascii_letters + string.digits) for _ in range(8))
                try:
                    db.execute('INSERT INTO snippets VALUES (?,?,?,?,?,?,?)',
                               (snippet_id, content, title, language, size, now, now+ttl_seconds))
                    break
                except sqlite3.IntegrityError:
                    continue
            else:
                raise ServiceError(503, 'unable to allocate unique ID')
            response = dict(id=snippet_id, url=f'{BASE_URL}/s/{snippet_id}',
                            created_at=self._iso(now), expires_at=self._iso(now+ttl_seconds))
            if key:
                db.execute('INSERT INTO idempotency_keys VALUES (?,?,?,?)',
                           (key, fingerprint, json.dumps(response), now))
            db.commit()
            return response

    def get(self, snippet_id):
        with self._connect() as db:
            row = db.execute('SELECT * FROM snippets WHERE id=? AND expires_at>?',
                             (snippet_id, time.time())).fetchone()
        if row is None:
            raise ServiceError(404, 'snippet not found or expired')
        return dict(id=row['id'], content=row['content'], title=row['title'],
                    language=row['language'], created_at=self._iso(row['created_at']),
                    expires_at=self._iso(row['expires_at']))

    def cleanup(self):
        with self._lock, self._connect() as db:
            result = db.execute('DELETE FROM snippets WHERE id IN (SELECT id FROM snippets WHERE expires_at <= ? LIMIT 1000)', (time.time(),))
            db.commit()
            return result.rowcount

service = PastebinService(DB_PATH)
print('Storage ready:', DB_PATH)

### Разбор блока 3: хранение и общая бизнес-логика

**Задача блока:** создать один сервис, который хранит записи и применяет правила независимо от способа обращения. Позже и REST, и gRPC будут вызывать один объект `service`. Сниппет здесь — сохранённый фрагмент текста; операции изменения его содержимого нет.

**1. `ServiceError`.** Собственное исключение хранит числовой статус и понятное сообщение. `super().__init__(message)` передаёт сообщение базовому классу `Exception`. Сервис сообщает об ошибке, а REST- и gRPC-обработчики позже переводят её в свой формат ответа.

**2. Инициализация `PastebinService`.** `__init__` сохраняет путь и создаёт `threading.RLock()` — блокировку для согласования операций записи из разных потоков этого объекта. `PRAGMA journal_mode=WAL` включает журнал SQLite, который облегчает совместную работу чтения и записи, но не делает количество параллельных писателей неограниченным.

Таблица `snippets` хранит уникальный `id`, текст, заголовок, язык, размер в байтах и две временные метки. `PRIMARY KEY` запрещает повтор ID, `NOT NULL` запрещает SQL-значение `NULL`. Индекс по `expires_at` помогает искать истёкшие записи. Таблица `idempotency_keys` хранит уникальный ключ запроса, отпечаток его параметров, ранее сформированный JSON-ответ и время создания. `IF NOT EXISTS` позволяет повторно открыть уже подготовленную базу.

**3. Вспомогательные методы.** `_connect()` открывает SQLite-соединение с ожиданием блокировки до 5 секунд. `check_same_thread=False` снимает проверку использования соединения только создавшим его потоком; само по себе это не заменяет синхронизацию. `sqlite3.Row` позволяет обращаться к столбцам по имени, например `row['content']`.

Конструкция `with ... as db` управляет транзакцией соединения: при нормальном выходе фиксирует её, при исключении откатывает. У SQLite этот контекстный менеджер **не закрывает само соединение**. `_iso()` переводит Unix-время в строку UTC с окончанием `Z`. Декоратор `@staticmethod` означает, что этому методу не нужен `self`. Начальный `_` в именах методов — соглашение о внутреннем использовании.

**4. Проверка данных в `create()`.** Значения после `=` в сигнатуре — аргументы по умолчанию. Дальше проверки идут последовательно:

- `content` должен быть строкой. `encode('utf-8')` превращает её в байты, а `len(...)` считает размер: один символ может занимать несколько байтов. Размер больше 1 МиБ даёт 413, пустая строка — 422. Строка из пробелов здесь считается непустой.
- TTL должен пройти проверку типа `int` и находиться в диапазоне от 60 секунд до 30 дней включительно, иначе возвращается 422.
- Заголовок должен быть строкой длиной до 200 символов, язык — строкой длиной от 1 до 32 символов. Эти длины считаются в символах, а не байтах.
- Если ключ передан, `uuid.UUID(key)` проверяет его формат, а `str(...)` приводит к единому представлению. Неподходящий ключ даёт 422.

**5. Идемпотентность.** Идемпотентный повтор позволяет безопасно отправить тот же запрос ещё раз и получить тот же результат без второй записи. Словарь `payload` содержит текст, заголовок, язык и TTL. `json.dumps(..., sort_keys=True, ensure_ascii=False)` сериализует его с устойчивым порядком ключей, `hashlib.sha256(...).hexdigest()` вычисляет отпечаток. Это не шифрование текста; отпечаток нужен для сравнения параметров.

После захвата блокировки выполняется `BEGIN IMMEDIATE`: SQLite начинает транзакцию с намерением записи. Если ключ уже сохранён, одинаковый отпечаток возвращает прежний ответ через `json.loads(...)`, а другой отпечаток даёт 409. Например, повтор с текстом `first` вернёт прежний ID, но замена текста на `different` с тем же ключом вызовет конфликт. Сравниваются все поля `payload`, включая TTL, а не только содержимое.

**6. Создание записи.** `time.time()` даёт текущее Unix-время в секундах. Генератор собирает ID из 8 случайных латинских букв и цифр через `secrets.choice`. Запрос `INSERT` передаёт значения отдельно через `?`: это параметризованный SQL, а не вставка пользовательского текста в команду.

Если возникает `sqlite3.IntegrityError`, цикл пробует ещё раз; всего предусмотрено три попытки. `else` у `for` выполняется, если цикл завершился без `break`: тогда выдаётся 503. При успехе формируется словарь с ID, публичным URL и датами. Если есть ключ, его отпечаток и ответ сохраняются в той же транзакции; `commit()` фиксирует запись и ключ вместе.

**7. Чтение и очистка.** `get()` ищет ID только с `expires_at > текущее время`. Поэтому запись с истёкшим TTL уже недоступна, даже если физически остаётся в базе. Отсутствующая или истёкшая запись даёт 404; найденная превращается в словарь с текстом и метаданными.

`cleanup()` удаляет до 1000 истёкших сниппетов за вызов и возвращает число удалённых строк. Этот метод только объявлен: автоматического расписания очистки в ноутбуке нет. Таблицу ключей он не очищает, поэтому повтор с прежним ключом может вернуть старый ответ даже после истечения TTL сниппета.

Последняя строка `service = PastebinService(DB_PATH)` создаёт общий экземпляр и подготавливает базу. Здесь находится основная логика; сетевые слои лишь принимают запросы и вызывают её.


**Самые важные методы:** `create()` — проверка и атомарное создание; `get()` — чтение с проверкой срока; `cleanup()` — физическое удаление. `_connect()` и `_iso()` — вспомогательные методы.

**Вопрос преподавателя:** «Что произойдёт при двух запросах с одним ключом?» — Локальная блокировка, транзакция `BEGIN IMMEDIATE` и уникальный ключ согласуют запись. После первого успешного запроса второй получает сохранённый ответ при совпадении параметров или 409 при несовпадении. `RLock` работает только в рамках данного объекта/процесса; ограничения SQLite действуют на уровне базы.

**Вопрос:** «SHA-256 шифрует текст?» — Нет, это отпечаток нормализованного запроса для обнаружения изменения параметров. Текст хранится в базе открыто.

**Вопрос:** «TTL закончился, но строка осталась — это ошибка?» — Нет: `get()` уже скрывает её, а `cleanup()` удаляет физически отдельным вызовом. Ключи идемпотентности этим методом не очищаются.


## 4. REST/JSON server (FastAPI)
Same business operations as the gRPC service. REST is the baseline; HTTP errors map to appropriate status codes.

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import PlainTextResponse
from pydantic import BaseModel, Field
import uvicorn

app = FastAPI(title='SIS2 Pastebin REST baseline')

class CreateBody(BaseModel):
    content: str
    title: str = ''
    language: str = 'plaintext'
    ttl_seconds: int = DEFAULT_TTL

def rest_call(fn, *args, **kwargs):
    try: return fn(*args, **kwargs)
    except ServiceError as e: raise HTTPException(status_code=e.status, detail=e.message)

@app.post('/v1/snippets', status_code=201)
def create_rest(body: CreateBody, idempotency_key: str | None = Header(default=None)):
    return rest_call(service.create, body.content, body.title, body.language, body.ttl_seconds, idempotency_key)

@app.get('/v1/snippets/{snippet_id}')
def get_rest(snippet_id: str):
    return rest_call(service.get, snippet_id)

@app.get('/s/{snippet_id}', response_class=PlainTextResponse)
def get_public(snippet_id: str):
    return rest_call(service.get, snippet_id)['content']

rest_server = None
rest_thread = None

def start_rest():
    global rest_server, rest_thread
    if rest_server is not None and rest_server.started:
        return
    config = uvicorn.Config(app, host='127.0.0.1', port=REST_PORT, log_level='warning', access_log=False)
    rest_server = uvicorn.Server(config)
    rest_thread = threading.Thread(target=rest_server.run, daemon=True)
    rest_thread.start()
    for _ in range(100):
        if rest_server.started: return
        time.sleep(.05)
    raise RuntimeError('REST server did not start')

start_rest()
print('REST listening at', BASE_URL, '| docs:', BASE_URL+'/docs')

### Разбор блока 4: REST-сервер на FastAPI

**Задача блока:** предоставить общий сервис через HTTP. Для запросов и обычных ответов используется JSON, а публичный маршрут возвращает простой текст.

**1. Приложение и тело запроса.** `FastAPI(...)` создаёт приложение, `title` задаёт его название в документации. `CreateBody(BaseModel)` — модель Pydantic: поле `content` обязательно, остальные получают указанные значения по умолчанию. FastAPI разбирает JSON и запускает проверку модели до вызова обработчика; затем правила размера и TTL проверяет `service.create`. Pydantic также может приводить некоторые входные значения к объявленным типам. Импорт `Field` в этом блоке не используется.

**2. Перевод ошибок.** `rest_call(fn, *args, **kwargs)` принимает функцию и её аргументы. `*args` собирает позиционные аргументы, `**kwargs` — именованные; в `fn(*args, **kwargs)` они передаются дальше. Если сервис выбрасывает `ServiceError`, обёртка создаёт `HTTPException` с тем же статусом и сообщением в поле `detail`. Здесь перехватываются именно ошибки сервиса, а не любые исключения Python.

**3. Маршруты.** Декоратор `@app...` связывает HTTP-метод и URL с функцией под ним.

| Запрос | Результат |
|---|---|
| `POST /v1/snippets` | Создаёт запись через `service.create`, возвращает JSON с ID, URL и датами; успешный статус — 201. |
| `GET /v1/snippets/{snippet_id}` | Передаёт ID в `service.get`, возвращает JSON с текстом и метаданными; успешный статус — 200. |
| `GET /s/{snippet_id}` | Берёт из результата `get()` только `content`, возвращает его как `text/plain`. Это публичный URL из ответа создания. |

В `create_rest` параметр `body` берётся из JSON, а `idempotency_key` — из HTTP-заголовка `Idempotency-Key`: `Header` по умолчанию преобразует подчёркивания имени в дефисы. Аннотация `str | None` допускает строку или отсутствие значения. В URL фигурные скобки означают переменную часть пути. Для успешного идемпотентного повтора POST тоже возвращается 201, поскольку статус задан на маршруте.

**4. Запуск Uvicorn.** `rest_server` и `rest_thread` сначала равны `None`. `global` позволяет функции `start_rest()` заменить эти переменные на уровне ноутбука. Проверка `rest_server.started` пропускает повторный запуск, если известный функции сервер уже работает.

`uvicorn.Config` задаёт приложение, локальный адрес `127.0.0.1`, порт и настройки логов. Uvicorn обслуживает FastAPI. Сервер запускается через `threading.Thread(..., daemon=True)`, чтобы длительная работа сервера не удерживала выполнение ячейки; фоновый поток не мешает завершению процесса Python.

После запуска цикл до 100 раз проверяет `started`, делая паузы по 0,05 секунды — это примерно 5 секунд ожидания. Если готовность не наступила, возникает `RuntimeError`. Затем печатаются адрес API и адрес `/docs`, где FastAPI предоставляет интерактивную документацию. Флаг `daemon` не заменяет корректную остановку из последней ячейки.


**Самые важные методы:** `create_rest()`, `get_rest()`, `get_public()` — транспортные адаптеры; `rest_call()` переводит ошибки; `start_rest()` запускает Uvicorn.

**Вопрос преподавателя:** «Чем `/v1/snippets/{id}` отличается от `/s/{id}`?» — Первый возвращает JSON с метаданными и текстом, второй — только текст. Чтение в обоих случаях выполняет один `service.get()`.

**Вопрос:** «Почему при повторном запуске появляется `address already in use`?» — На порту уже работает предыдущий сервер. Кроме того, эта ячейка заново присваивает `rest_server = None`, поэтому повтор всей ячейки теряет ссылку на старый объект; защита внутри `start_rest()` этого не предотвращает. Надёжный полный перезапуск начинается с остановки/перезапуска ядра.


## 5. gRPC server
Protocol Buffers messages, the same storage and validation, and status mappings for errors.

In [ ]:
import grpc
from concurrent import futures

GRPC_ERROR_CODES = {
    404: grpc.StatusCode.NOT_FOUND,
    409: grpc.StatusCode.ALREADY_EXISTS,
    413: grpc.StatusCode.RESOURCE_EXHAUSTED,
    422: grpc.StatusCode.INVALID_ARGUMENT,
    503: grpc.StatusCode.UNAVAILABLE,
}

class PastebinGrpc(pb_grpc.PastebinServicer):
    def CreateSnippet(self, request, context):
        try:
            result = service.create(request.content, request.title, request.language or 'plaintext',
                                    request.ttl_seconds or DEFAULT_TTL, request.idempotency_key or None)
            return pb.CreateSnippetResponse(**result)
        except ServiceError as e:
            context.abort(GRPC_ERROR_CODES.get(e.status, grpc.StatusCode.INTERNAL), e.message)

    def GetSnippet(self, request, context):
        try:
            return pb.GetSnippetResponse(**service.get(request.id))
        except ServiceError as e:
            context.abort(GRPC_ERROR_CODES.get(e.status, grpc.StatusCode.INTERNAL), e.message)

grpc_server = grpc.server(futures.ThreadPoolExecutor(max_workers=10),
                          options=[('grpc.max_receive_message_length', 2 * MAX_BYTES),
                                   ('grpc.max_send_message_length', 2 * MAX_BYTES)])
pb_grpc.add_PastebinServicer_to_server(PastebinGrpc(), grpc_server)
if grpc_server.add_insecure_port(f'127.0.0.1:{GRPC_PORT}') == 0:
    raise RuntimeError('Could not bind gRPC port')
grpc_server.start()
print(f'gRPC listening at 127.0.0.1:{GRPC_PORT}')

### Разбор блока 5: сервер gRPC

**Задача блока:** открыть второй сетевой интерфейс к тому же объекту `service`. Здесь передаются сообщения Protocol Buffers и вызываются именованные методы, объявленные в `PROTO`.

**1. Соответствие ошибок.** У gRPC своя система статусов, поэтому `GRPC_ERROR_CODES` переводит числовые статусы сервиса:

| Статус сервиса | Статус gRPC | Причина в этом приложении |
|---|---|---|
| 404 | `NOT_FOUND` | Запись отсутствует или истекла. |
| 409 | `ALREADY_EXISTS` | Ключ повторно использован с другими параметрами. |
| 413 | `RESOURCE_EXHAUSTED` | Текст превышает допустимый размер. |
| 422 | `INVALID_ARGUMENT` | Данные не проходят проверку. |
| 503 | `UNAVAILABLE` | Не удалось выделить уникальный ID. |

**2. Реализация RPC.** `PastebinGrpc` наследуется от сгенерированного `PastebinServicer`. Имена `CreateSnippet` и `GetSnippet` соответствуют контракту. `request` содержит уже разобранное сообщение, `context` позволяет управлять текущим RPC, в том числе завершать его с ошибкой.

`CreateSnippet` извлекает поля и вызывает `service.create`. Выражение `x or значение` подставляет правую часть, если `x` ложно: пустой язык заменяется на `plaintext`, нулевой TTL — на 7 дней, пустой ключ — на `None`. Это связано со скалярными значениями по умолчанию в `proto3`. Важный нюанс: явно переданный TTL `0` здесь тоже станет значением по умолчанию, тогда как REST передаст ноль сервису и получит 422. Пустой язык также обрабатывается транспортами по-разному.

`pb.CreateSnippetResponse(**result)` разворачивает словарь в именованные аргументы конструктора сообщения. Например, ключ `id` становится аргументом `id=...`. `GetSnippet` аналогично передаёт ID в `service.get` и строит `GetSnippetResponse`.

При `ServiceError` вызов `context.abort(...)` прекращает RPC с подходящим статусом и текстом ошибки. `.get(e.status, grpc.StatusCode.INTERNAL)` задаёт запасной статус, если в таблице нет нужного кода.

**3. Создание и запуск сервера.** `ThreadPoolExecutor(max_workers=10)` задаёт пул до десяти рабочих потоков для обработчиков. Параметры `grpc.max_receive_message_length` и `grpc.max_send_message_length` разрешают сообщения до 2 МиБ, учитывая метаданные и служебные поля. Бизнес-лимит самого `content` остаётся 1 МиБ — его отдельно проверяет сервис.

`add_PastebinServicer_to_server(...)` регистрирует обработчик. `add_insecure_port(...)` привязывает сервер к локальному адресу без TLS; нулевой результат означает, что порт привязать не удалось. `grpc_server.start()` запускает обработку в фоне и возвращает управление ячейке.

Оба транспорта используют одну базу и общий сервис, поэтому созданную через gRPC запись можно читать через REST. Поле `url` у ответа gRPC содержит обычную HTTP-ссылку на публичный REST-маршрут.


**Самые важные методы:** серверные `CreateSnippet()` и `GetSnippet()`, `context.abort()` и регистрация `add_PastebinServicer_to_server()`.

**Вопрос преподавателя:** «Одинаковая ли логика REST и gRPC?» — Основные проверки и хранение общие, но транспортные значения по умолчанию различаются: gRPC заменяет нулевой TTL и пустой язык, REST передаёт их на проверку. Поэтому полную эквивалентность всех граничных входов этот код не гарантирует.

**Вопрос:** «Почему разрешены сообщения 2 МиБ, а текст ограничен 1 МиБ?» — Сетевое сообщение содержит не только текст. Транспортный предел и бизнес-ограничение проверяются на разных уровнях.

**Вопрос:** «Почему HTTP-ссылка работает для записи из gRPC?» — Общие сервис и база, а URL строится через REST-адрес. Формат создания записи не ограничивает способ её чтения.


## 6. Client examples and smoke tests
Both clients send equivalent requests. The gRPC service returns a browser-friendly REST URL.

In [ ]:
import httpx

channel = grpc.insecure_channel(f'127.0.0.1:{GRPC_PORT}')
grpc.channel_ready_future(channel).result(timeout=5)
stub = pb_grpc.PastebinStub(channel)

key = str(uuid.uuid4())
request = pb.CreateSnippetRequest(content='print("Hello from SIS2")', title='demo',
                                  language='python', ttl_seconds=3600, idempotency_key=key)
created = stub.CreateSnippet(request, timeout=2)
created_again = stub.CreateSnippet(request, timeout=2)
assert created.id == created_again.id, 'idempotency failed'
read_grpc = stub.GetSnippet(pb.GetSnippetRequest(id=created.id), timeout=2)
assert read_grpc.content == request.content

with httpx.Client(timeout=2) as client:
    rest_response = client.post(f'{BASE_URL}/v1/snippets', json={
        'content': 'print("Hello from REST")', 'title': 'demo',
        'language': 'python', 'ttl_seconds': 3600},
        headers={'Idempotency-Key': str(uuid.uuid4())})
    assert rest_response.status_code == 201, rest_response.text
    rest_created = rest_response.json()
    rest_read = client.get(f'{BASE_URL}/v1/snippets/{rest_created["id"]}')
    assert rest_read.status_code == 200
    assert rest_read.json()['content'] == 'print("Hello from REST")'
    public_read = client.get(f'{BASE_URL}/s/{rest_created["id"]}')
    assert public_read.text == 'print("Hello from REST")'

print('PASS: gRPC create, gRPC read, idempotent retry, REST create/read, public URL')
print('gRPC ID:', created.id, '| REST ID:', rest_created['id'])
print('Public URL:', rest_created['url'])

### Разбор блока 6: клиенты и базовые проверки

**Задача блока:** обратиться к работающим серверам по сети и проверить основной сценарий «создать → прочитать», повтор запроса и публичную ссылку. Для выполнения нужны успешно запущенные предыдущие ячейки.

**1. Подключение gRPC.** `grpc.insecure_channel(...)` создаёт канал без TLS. `channel_ready_future(channel).result(timeout=5)` ждёт готовности канала до 5 секунд. `PastebinStub(channel)` — сгенерированный клиентский объект, через который вызываются RPC, например `stub.CreateSnippet(...)`.

**2. Запрос создания.** `uuid.uuid4()` создаёт случайный UUID для ключа идемпотентности. В `CreateSnippetRequest` передаются пример текста, заголовок, язык Python и TTL 3600 секунд, то есть один час. Текст `print(...)` хранится как строка: сервис его не исполняет.

Один запрос отправляется дважды. `assert created.id == created_again.id` проверяет одинаковый ID, то есть поведение идемпотентного повтора. Затем `GetSnippetRequest` передаёт ID, а следующая проверка сравнивает прочитанный текст с исходным. `timeout=2` ограничивает время ожидания каждого RPC; автоматического повторения после ошибки в этом коде нет — второй вызов написан явно.

**3. Запросы REST через HTTPX.** Контекст `with httpx.Client(timeout=2) as client` открывает переиспользуемый HTTP-клиент и закрывает его при выходе. Параметр `json={...}` сериализует словарь в JSON-тело, `headers={...}` передаёт ключ в HTTP-заголовке. HTTPX применяет таймауты к сетевым операциям; это не обязательно общий предел в 2 секунды на весь сценарий.

POST создаёт другую запись и должен вернуть 201. `rest_response.json()` разбирает тело ответа в Python-структуру, откуда берётся ID. Далее GET проверяет статус 200 и содержимое JSON, а запрос `/s/{id}` — текст публичной страницы. Здесь gRPC- и REST-примеры используют разные строки и разные ключи, поэтому у них отдельные записи.

**4. Как читать результат.** `assert условие, сообщение` выбрасывает `AssertionError`, если условие ложно; в обычном режиме выполнение ячейки на этом месте останавливается. У первой REST-проверки сообщением служит `rest_response.text`, чтобы видеть причину неожиданного статуса. Если выполнение дошло до `PASS`, все предшествующие утверждения прошли; затем печатаются ID и URL.

Это базовые проверки работоспособности, а не измерение скорости или устойчивости под нагрузкой. Наличие таймаутов не означает, что проверен сценарий их срабатывания; отдельные проверки истечения TTL и сетевых сбоев здесь отсутствуют.


**Главные вызовы:** `stub.CreateSnippet()`, `stub.GetSnippet()`, HTTPX `post()` / `get()` и проверки `assert`.

**Вопрос преподавателя:** «Как доказано, что повтор не создал новую запись?» — Здесь сравниваются ID двух ответов. Позже отдельная проверка дополнительно считает строки в базе; одной проверки ID недостаточно для всех возможных ошибок реализации.

**Вопрос:** «Это уже тест автоматического повтора при сетевом сбое?» — Нет. Здесь явно отправлены два успешных запроса. Управляемые сбои и автоматические повторы проверяются в разделе 9.


## 7. Additional functional checks
Validation, duplicate-key conflict, and missing IDs. **Step 4** will add dedicated timeout/retry tests; **Step 5** will add latency and payload-size benchmarks.

In [ ]:
from fastapi.testclient import TestClient

with TestClient(app) as client:
    assert client.get('/v1/snippets/NOTFOUND').status_code == 404
    assert client.post('/v1/snippets', json={'content': '', 'ttl_seconds': 60}).status_code == 422
    assert client.post('/v1/snippets', json={'content': 'a' * (MAX_BYTES + 1), 'ttl_seconds': 60}).status_code == 413
    conflict_key = str(uuid.uuid4())
    h = {'Idempotency-Key': conflict_key}
    a = client.post('/v1/snippets', json={'content': 'first', 'ttl_seconds': 60}, headers=h)
    b = client.post('/v1/snippets', json={'content': 'different', 'ttl_seconds': 60}, headers=h)
    assert a.status_code == 201 and b.status_code == 409
print('PASS: 404, empty content, oversize content, conflicting idempotency key')

### Разбор блока 7: проверки ошибочных запросов

**Задача блока:** убедиться, что REST API возвращает ожидаемые ошибки и защищает ключ идемпотентности от использования с другим запросом.

`TestClient(app)` обращается непосредственно к FastAPI-приложению внутри процесса, без сетевого запроса на порт Uvicorn. При этом обработчики и объект `service` настоящие, поэтому проверки работают с той же SQLite-базой. `with` управляет жизненным циклом тестового клиента.

| Проверка | Ожидаемый результат и причина |
|---|---|
| GET с ID `NOTFOUND` | 404: запись с таким ID не найдена. |
| POST с пустым `content` | 422: общий сервис запрещает пустой текст. TTL 60 допустим. |
| POST с `'a' * (MAX_BYTES + 1)` | 413: каждый ASCII-символ `a` занимает один байт, поэтому текст превышает лимит ровно на байт. |
| Два POST с одним ключом, но текстами `first` и `different` | Первый возвращает 201, второй — 409: ключ уже связан с другим отпечатком параметров. |

В проверке конфликта `conflict_key` — новый UUID, а `h` — словарь заголовков, который передаётся в оба запроса. Переменные `a` и `b` содержат HTTP-ответы. Выражение `a.status_code == 201 and b.status_code == 409` требует, чтобы оба условия выполнялись.

Итоговая строка `PASS` печатается только после успешного выполнения всех `assert`. Эти проверки дополняют успешные сценарии предыдущего блока, но напрямую проверяют ошибки только REST. В этом блоке нет отдельных проверок ошибок gRPC, границ TTL, истечения срока или производительности; таймауты/повторы при сбое и бенчмарки указаны в ноутбуке как дальнейшая работа.


**Главный инструмент:** `TestClient` — проверяет HTTP-обработчики без сетевого соединения, но с настоящим сервисом и базой.

**Вопрос преподавателя:** «Почему ошибка клиента возвращает 422, а конфликт — 409?» — 422 означает неподходящие данные, 409 — столкновение нового запроса с уже сохранённым состоянием ключа. Повторять их автоматически с теми же данными бессмысленно.

**Вопрос:** «Проверена ли граница размера и TTL полностью?» — Нет. Проверены пустой текст и превышение размера на байт; отдельных утверждений для ровно 1 МиБ, TTL 59/60 и максимального TTL в этой ячейке нет.


## 8. Reliability: idempotency, deadlines, and retries (Step 4)

**Idempotency:** The storage layer already writes the snippet and the idempotency record in one SQLite transaction. Repeating the same UUID + payload returns the same ID; reusing the UUID with different data raises a conflict. The REST and gRPC interfaces share this behavior.

**Timeouts:** Clients set a 140 ms overall operation budget (SIS #1 design target); a request's remaining budget is passed as a per-attempt HTTP timeout or gRPC deadline. The original SIS #1's Redis (50 ms) and database read (80 ms) timeouts are **not implemented**, since this local prototype uses SQLite without Redis.

**Retries:** At most three total attempts for temporary failures, exponential backoff + random jitter, respecting the same overall deadline. Creation must reuse the *same idempotency key* across attempts. Retry only HTTP 429/503/504, gRPC UNAVAILABLE/RESOURCE_EXHAUSTED/DEADLINE_EXCEEDED, and transport timeouts. For the 140 ms budget, backoff is intentionally small; a 2 s budget is useful for interactive smoke tests.

**Note:** The notebook's original `stub` and `channel` must still be running. Execute this section before the shutdown cell.

In [ ]:
import random
from dataclasses import dataclass

RETRYABLE_HTTP = {429, 503, 504}
RETRYABLE_GRPC = {grpc.StatusCode.UNAVAILABLE, grpc.StatusCode.RESOURCE_EXHAUSTED,
                  grpc.StatusCode.DEADLINE_EXCEEDED}

@dataclass
class RetryStats:
    attempts: int = 0
    elapsed_ms: float = 0.0

def call_with_retries(operation, retryable, *, budget_s=0.140, max_attempts=3,
                      base_backoff_s=0.010, rng=None, sleep=time.sleep, clock=time.monotonic):
    """operation(remaining_seconds) -> result; never start an attempt after deadline."""
    if budget_s <= 0 or max_attempts < 1:
        raise ValueError('budget_s must be positive; max_attempts >= 1')
    rng = rng or random.Random()
    start = clock()
    deadline = start + budget_s
    stats = RetryStats()
    for attempt in range(max_attempts):
        remaining = deadline - clock()
        if remaining <= 0:
            raise TimeoutError('overall operation deadline exceeded')
        stats.attempts += 1
        try:
            value = operation(remaining)
            stats.elapsed_ms = (clock() - start) * 1000
            return value, stats
        except Exception as exc:
            if not retryable(exc) or attempt + 1 >= max_attempts:
                raise
            remaining = deadline - clock()
            if remaining <= 0:
                raise TimeoutError('overall operation deadline exceeded') from exc
            backoff = base_backoff_s * (2 ** attempt) * (0.5 + rng.random())
            # Do not spend the entire remaining budget sleeping.
            delay = min(backoff, remaining / 2)
            if delay > 0:
                sleep(delay)
    raise AssertionError('unreachable')

def grpc_retryable(exc):
    return isinstance(exc, grpc.RpcError) and exc.code() in RETRYABLE_GRPC

def http_retryable(exc):
    return (isinstance(exc, httpx.HTTPStatusError) and
            exc.response.status_code in RETRYABLE_HTTP) or isinstance(
            exc, (httpx.TimeoutException, httpx.TransportError))

# Same request object/key across all attempts: mandatory for safe create retries.
def create_grpc_reliable(content, *, title='', language='plaintext', ttl_seconds=3600,
                         key=None, budget_s=0.140, grpc_stub=None):
    key = key or str(uuid.uuid4())
    req = pb.CreateSnippetRequest(content=content, title=title, language=language,
                                  ttl_seconds=ttl_seconds, idempotency_key=key)
    active_stub = grpc_stub or stub
    return call_with_retries(lambda remaining: active_stub.CreateSnippet(req, timeout=remaining),
                             grpc_retryable, budget_s=budget_s)

def get_grpc_reliable(snippet_id, *, budget_s=0.140, grpc_stub=None):
    active_stub = grpc_stub or stub
    req = pb.GetSnippetRequest(id=snippet_id)
    return call_with_retries(lambda remaining: active_stub.GetSnippet(req, timeout=remaining),
                             grpc_retryable, budget_s=budget_s)

def create_rest_reliable(content, *, title='', language='plaintext', ttl_seconds=3600,
                          key=None, budget_s=0.140, client=None):
    key = key or str(uuid.uuid4())
    payload = dict(content=content, title=title, language=language, ttl_seconds=ttl_seconds)
    def perform(remaining):
        def send(c):
            response = c.post(f'{BASE_URL}/v1/snippets', json=payload,
                              headers={'Idempotency-Key': key}, timeout=remaining)
            response.raise_for_status()
            return response.json()
        if client is not None:
            return send(client)
        with httpx.Client() as c:
            return send(c)
    return call_with_retries(perform, http_retryable, budget_s=budget_s)

def get_rest_reliable(snippet_id, *, budget_s=0.140, client=None):
    def perform(remaining):
        def send(c):
            response = c.get(f'{BASE_URL}/v1/snippets/{snippet_id}', timeout=remaining)
            response.raise_for_status()
            return response.json()
        if client is not None:
            return send(client)
        with httpx.Client() as c:
            return send(c)
    return call_with_retries(perform, http_retryable, budget_s=budget_s)

print('Reliable clients ready: 140 ms budget, up to 3 attempts')

### Разбор: надёжные клиенты, общий дедлайн и повторные попытки

**Роль блока.** Отделить правило повторов от конкретного транспорта. Обычные клиентские вызовы оборачиваются в общий алгоритм `call_with_retries()`. `RetryStats` — dataclass со счётчиком попыток и временем успешной операции.

**1. Политика ошибок.** `RETRYABLE_HTTP` содержит 429, 503 и 504. Для gRPC выбраны `UNAVAILABLE`, `RESOURCE_EXHAUSTED`, `DEADLINE_EXCEEDED`. Предикаты `grpc_retryable()` и `http_retryable()` определяют, можно ли повторить конкретное исключение. HTTP-клиент должен вызвать `raise_for_status()`, иначе ответ 503 останется обычным объектом и не попадёт в обработку исключений. HTTPX дополнительно повторяет транспортные ошибки/таймауты.

**2. `call_with_retries()` — центральный метод этого раздела.** Он принимает `operation(remaining_seconds)` и функцию проверки ошибки. Звёздочка в сигнатуре делает дальнейшие настройки именованными аргументами. По умолчанию бюджет 0,140 секунды и максимум **три попытки суммарно**, а не три повтора после первой.

Сначала вычисляется `deadline = start + budget_s`. Перед каждой попыткой считается остаток времени; при его исчерпании новая попытка не начинается. `time.monotonic()` подходит для длительностей: он не является календарными часами и не должен идти назад при их корректировке.

Если вызов успешен, возвращаются `(value, stats)`. При неповторяемой ошибке или последней попытке `raise` передаёт исходное исключение наружу. Иначе пауза растёт по формуле `base_backoff_s * 2**attempt`, умноженной на случайный коэффициент от 0,5 до 1,5. Это exponential backoff с jitter: клиенты не повторяют синхронно. `min(backoff, remaining / 2)` оставляет часть бюджета следующей попытке. `raise ... from exc` сохраняет причинную связь исключений.

**3. Адаптеры транспорта.** `create_grpc_reliable()` и `create_rest_reliable()` создают ключ и тело **до** цикла повторов; следовательно, все попытки одной операции используют одни данные. `get_grpc_reliable()` и `get_rest_reliable()` читают запись. `lambda remaining: ...` — короткая функция, которая передаёт остаток бюджета RPC. Вложенные `perform()` / `send()` в REST выполняют аналогичную работу и позволяют подставить тестовый клиент.

**Граница гарантии:** алгоритм не начинает попытку после дедлайна и передаёт остаток транспорту, но не может принудительно остановить произвольную функцию, игнорирующую таймаут. HTTPX ограничивает отдельные сетевые фазы; это не строгий общий дедлайн для всей HTTP-операции. Нет и проверки бюджета после успешного возвращения `operation`. Поэтому «всегда завершится ровно за 140 мс» здесь утверждать нельзя. `elapsed_ms` присваивается при успехе, а при исключении статистика наружу не возвращается.

**Самые важные методы:** `call_with_retries()`, `create_grpc_reliable()` и `create_rest_reliable()` — основа надёжности клиента.

**Вопрос преподавателя:** «Почему нельзя генерировать новый UUID при каждом повторе POST?» — Сервер воспримет попытки как разные создания; при потере первого ответа появятся дубликаты.

**Вопрос:** «Почему нельзя повторять любую ошибку?» — Проверка данных не исправится от ожидания; повторы создают дополнительную нагрузку. Даже `RESOURCE_EXHAUSTED` может означать постоянное превышение размера, и в этом прототипе его повтор не поможет: политика нуждается в уточнении для продакшена.

**Вопрос:** «Сервер мог сохранить запись, хотя клиент получил таймаут?» — Да. Таймаут ожидания не доказывает откат серверной операции; от повторного создания защищает сохранённый ключ.


## 9. Reliability tests

The following tests check **actual SQLite idempotency**, real gRPC and REST calls, injected transient HTTP errors, and a real gRPC deadline on a deliberately slow test server. Tests use a larger 2 s budget for normal smoke calls to avoid environment-specific notebook startup delays; the 140 ms budget is tested separately as a deadline.

In [ ]:
# 9a. Real service: same key + same payload -> same record; changed payload -> conflict.
test_key = str(uuid.uuid4())
a = service.create('step4 idempotency test', ttl_seconds=3600, key=test_key)
b = service.create('step4 idempotency test', ttl_seconds=3600, key=test_key)
assert a == b
try:
    service.create('different payload', ttl_seconds=3600, key=test_key)
    raise AssertionError('Expected 409')
except ServiceError as e:
    assert e.status == 409
with service._connect() as db:
    assert db.execute('SELECT COUNT(*) FROM snippets WHERE id=?', (a['id'],)).fetchone()[0] == 1
print('PASS: atomic idempotency and key/payload conflict')

# 9b. Real servers with reliable clients.
g_created, gs = create_grpc_reliable('reliable gRPC example', budget_s=2)
g_read, grs = get_grpc_reliable(g_created.id, budget_s=2)
assert g_read.content == 'reliable gRPC example'
r_created, rs = create_rest_reliable('reliable REST example', budget_s=2)
r_read, rrs = get_rest_reliable(r_created['id'], budget_s=2)
assert r_read['content'] == 'reliable REST example'
print('PASS: gRPC and REST reliable create/get')
print('Attempts:', {'grpc_create': gs.attempts, 'rest_create': rs.attempts})

### Разбор: проверка идемпотентности и реальных надёжных клиентов

**Порядок проверки.** Сначала общий сервис дважды получает одинаковые данные с одним UUID. `assert a == b` сравнивает все поля ответов. Затем изменённый текст с тем же ключом обязан вызвать `ServiceError(409)`. Явный `raise AssertionError('Expected 409')` делает тест провальным, если ожидаемое исключение не возникло.

SQL `COUNT(*) ... WHERE id=?` подтверждает, что по полученному ID хранится одна запись. Это последовательный тест: он проверяет результат повторов, но не моделирует одновременную гонку запросов. Атомарность обеспечивается структурой транзакции; нагрузочного доказательства конкурентного поведения здесь нет.

Во второй части настоящие gRPC- и REST-серверы вызываются через надёжные обёртки. Возвращённый кортеж распаковывается в результат и статистику: например, `g_created, gs`. Тексты после чтения сверяются с исходными, затем печатается число попыток.

**Важно:** здесь задан `budget_s=2`, то есть две секунды, чтобы не делать базовый функциональный тест зависимым от строгого бюджета 140 мс. Прямой вызов `service.create()` и сетевой вызов проверяют разные уровни.

**Главные методы:** `service.create()`, `create_*_reliable()` / `get_*_reliable()`, SQL-подсчёт записей.

**Вопрос преподавателя:** «Можно ли по одной строке `PASS: atomic idempotency` доказать отсутствие гонок?» — Нет. Название вывода шире самого теста: запросы здесь последовательные. Для гонок нужен отдельный конкурентный сценарий.

**Вопрос:** «Проверяет ли успешный вызов клиента обработку сбоев?» — Сам по себе нет. Для этого следующий блок преднамеренно возвращает ошибки.


In [ ]:
# 9c. HTTP retry injection: 503 twice, then 201; confirm SAME key and payload.
seen = []
def fake_handler(request):
    seen.append((request.headers.get('Idempotency-Key'), request.content))
    if len(seen) < 3:
        return httpx.Response(503, json={'detail': 'temporarily unavailable'})
    return httpx.Response(201, json={'id': 'test1234', 'url': 'http://example/s/test1234'})
with httpx.Client(transport=httpx.MockTransport(fake_handler)) as fake_client:
    result, stats = create_rest_reliable('retry example', key=str(uuid.uuid4()),
                                        budget_s=2, client=fake_client)
assert stats.attempts == 3 and len(set(k for k, _ in seen)) == 1
assert len(set(body for _, body in seen)) == 1
assert result['id'] == 'test1234'
print('PASS: retry 503 twice then success, key and body unchanged')

# Non-retryable HTTP 422 must stop after one attempt.
calls = []
def reject(request):
    calls.append(1)
    return httpx.Response(422, json={'detail': 'bad request'})
with httpx.Client(transport=httpx.MockTransport(reject)) as bad_client:
    try:
        create_rest_reliable('bad', budget_s=2, client=bad_client)
        raise AssertionError('Expected HTTP 422')
    except httpx.HTTPStatusError as exc:
        assert exc.response.status_code == 422
assert len(calls) == 1
print('PASS: validation errors are not retried')

### Разбор: управляемые HTTP-сбои и проверка повторов

**Роль блока.** Получить предсказуемую последовательность ошибок без остановки настоящего сервера. `httpx.MockTransport(fake_handler)` направляет запрос в Python-функцию вместо сети.

`fake_handler()` сохраняет ключ и исходные байты тела в `seen`. Первые два вызова возвращают 503, третий — 201 с тестовым ID. Надёжная обёртка обязана сделать ровно три попытки. Генератор в `set(k for k, _ in seen)` собирает уникальные ключи; размер множества 1 означает одинаковый ключ на всех попытках. Аналогично сравниваются байты тела. `_` обозначает ненужный элемент пары.

Затем обработчик `reject()` всегда возвращает 422. Клиент должен пробросить `HTTPStatusError`, а список `calls` должен содержать ровно один вызов. Так проверяется, что ошибка валидации не приводит к повторам.

**Главные методы:** `MockTransport()`, `fake_handler()`, `reject()`, а проверяемый алгоритм — `call_with_retries()` через REST-обёртку.

**Вопрос преподавателя:** «Запись `test1234` появилась в SQLite?» — Нет: это искусственный ответ тестового транспорта; общий сервис не вызывался, а адрес `http://example/...` — фиктивный.

**Вопрос:** «Что тест не доказывает?» — Реальные сетевые задержки, разрыв TCP, серверную конкуренцию и соблюдение 140 мс: здесь бюджет 2 секунды. Тест точно проверяет выбранную последовательность повторов и постоянство ключа/тела.


In [ ]:
# 9d. Actual gRPC deadline: temporary server sleeps longer than allowed.
class SlowPastebin(pb_grpc.PastebinServicer):
    def GetSnippet(self, request, context):
        time.sleep(0.25)
        return pb.GetSnippetResponse(id=request.id, content='late')
    def CreateSnippet(self, request, context):
        time.sleep(0.25)
        return pb.CreateSnippetResponse(id='late')

slow_server = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
pb_grpc.add_PastebinServicer_to_server(SlowPastebin(), slow_server)
slow_port = slow_server.add_insecure_port('127.0.0.1:0')
assert slow_port > 0
slow_server.start()
slow_channel = grpc.insecure_channel(f'127.0.0.1:{slow_port}')
try:
    slow_stub = pb_grpc.PastebinStub(slow_channel)
    start = time.monotonic()
    try:
        get_grpc_reliable('anything', budget_s=0.140, grpc_stub=slow_stub)
        raise AssertionError('Expected deadline exceeded')
    except (grpc.RpcError, TimeoutError) as exc:
        if isinstance(exc, grpc.RpcError):
            assert exc.code() == grpc.StatusCode.DEADLINE_EXCEEDED
    elapsed = time.monotonic() - start
    assert elapsed < 0.5, f'Unexpectedly slow deadline enforcement: {elapsed:.3f}s'
    print(f'PASS: gRPC deadline enforced (elapsed {elapsed*1000:.1f} ms)')
finally:
    slow_channel.close()
    slow_server.stop(grace=0).wait(timeout=2)

### Разбор: реальный gRPC-дедлайн на медленном сервере

**Постановка.** Временный `SlowPastebin` намеренно ждёт 250 мс, а клиенту разрешено только 140 мс. В обоих RPC объявлено ожидание, но в этом тесте вызывается только `GetSnippet`.

Сервер привязывается к `127.0.0.1:0`: нулевой порт означает, что ОС выберет свободный порт. Возвращённое число сохраняется в `slow_port`. Отдельный канал и stub позволяют передать тестовый сервер в `get_grpc_reliable(..., grpc_stub=slow_stub)`, не меняя основной сервис.

`time.monotonic()` измеряет всё клиентское ожидание. Вложенный `try/except` требует `DEADLINE_EXCEEDED` для ошибки gRPC либо `TimeoutError` от общей обёртки. Проверка `elapsed < 0.5` допускает накладные расходы и планирование потоков; она не требует завершения ровно на 140-й миллисекунде.

`finally` закрывает канал и останавливает временный сервер даже при провале утверждения. Это предотвращает оставшиеся фоновые ресурсы после теста.

**Главные методы:** `SlowPastebin.GetSnippet()`, `get_grpc_reliable()`, `stop()` и `close()`.

**Вопрос преподавателя:** «Прекратился ли Python-обработчик сразу при дедлайне клиента?» — Не обязательно: серверная функция здесь спит и не проверяет отмену. Клиент перестаёт ждать; это не автоматический откат побочных эффектов.

**Вопрос:** «Почему время меньше 500 мс, а не строго меньше 140 мс?» — У теста есть допуск на среду исполнения. Это проверка механизма дедлайна с запасом, а не жёсткая гарантия SLA.


## Step 4 result and limits

- **Idempotency:** implemented in the shared SQLite transaction and exercised above.
- **Timeouts:** per-attempt gRPC deadlines and HTTP client timeouts, bounded by a common overall operation budget. The server itself does not guarantee cancellation of in-progress SQLite work after the client times out; idempotency handles safe repeats.
- **Retries:** up to three total attempts, exponential backoff + jitter, only on transient failures, no retry on 4xx validation errors.
- **Scope:** local SQLite prototype, not a distributed PostgreSQL/Redis/CDN deployment. The next step measures real latency and serialized payload sizes against REST/JSON. Do not claim the SIS #1 p99 < 150 ms target is met without benchmarking.


## 10. Step 5 — Benchmark methodology

We measure **client-observed round-trip latency** for successful `CreateSnippet` and `GetSnippet` calls, using the same payloads and shared SQLite storage for gRPC and REST. A persistent HTTP client and gRPC channel are reused. Warm-up requests are excluded. Requests are sequential (concurrency = 1), so these measurements **do not establish 1,000 RPS capacity**.

Payload size is measured as serialized **application message bytes** (Protobuf `SerializeToString` vs compact UTF-8 JSON), not HTTP/2 frame or HTTP header bytes. Results will differ by machine and load. We use no retries in the timed loop so latency measures single attempts; Step 4 tested retry behavior separately.

In [ ]:
import csv, math, statistics, platform
from collections import defaultdict

# Start with a small repeatable test. Change to (100, 1000) for more samples.
BENCH_REQUESTS_PER_CASE = 100
BENCH_WARMUP = 10
BENCH_SIZES = (100, 1024, 10240)
BENCH_TIMEOUT_S = 5.0


def percentile(values, p):
    if not values:
        return float('nan')
    ordered = sorted(values)
    pos = (len(ordered) - 1) * p / 100
    lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] + (ordered[hi] - ordered[lo]) * (pos - lo)


def summarize(protocol, operation, size, times_ms, failures, request_bytes, response_bytes):
    return {
        'protocol': protocol, 'operation': operation, 'content_bytes': size,
        'attempts': len(times_ms) + failures, 'successes': len(times_ms),
        'failures': failures,
        'success_rate_pct': round(100 * len(times_ms) / (len(times_ms) + failures), 2),
        'mean_ms': round(statistics.mean(times_ms), 3) if times_ms else None,
        'p50_ms': round(percentile(times_ms, 50), 3) if times_ms else None,
        'p95_ms': round(percentile(times_ms, 95), 3) if times_ms else None,
        'p99_ms': round(percentile(times_ms, 99), 3) if times_ms else None,
        'request_bytes': request_bytes, 'response_bytes': response_bytes,
    }

print('Environment:', platform.platform(), '| Python:', platform.python_version())
print('Benchmark:', BENCH_REQUESTS_PER_CASE, 'measured calls per protocol / operation / size')

### Разбор: параметры эксперимента и расчёт статистики

**План.** Три размера текста: 100, 1024 и 10240 байт; два протокола; две операции. Получается `3 × 2 × 2 = 12` сценариев. На каждый приходится 10 прогревочных и 100 измеряемых вызовов. Следовательно, при полном прохождении будет 1200 измеряемых попыток и 120 прогревочных; подготовительные создания для Get считаются отдельно.

**`percentile(values, p)` — важный метод статистики.** Он сортирует значения и находит позицию `(n - 1) * p / 100`. Если позиция между элементами, результат линейно интерполируется. Для `[1, 2, 3, 4]` p50 равен 2,5. Пустой список даёт `NaN`.

p50 — медиана; p95 — оценка границы, около которой лежат 95% наблюдений; p99 — оценка более редких задержек. Это описание выборки, а не гарантия следующего запроса. При 100 измерениях p99 сильно зависит от нескольких самых медленных вызовов. Среднее чувствительно к выбросам, поэтому одного `mean_ms` недостаточно.

**`summarize()` — собирает строку итогового CSV.** `attempts` включает успешные и неуспешные измеряемые попытки; `success_rate_pct` — их доля успеха. Задержки считаются **только по успешным** вызовам. Когда их нет, значения задержки и размеров остаются `None`. Размеры — средние длины тел сообщений, а не сетевой трафик. `round()` округляет отчёт, но не улучшает точность измерения.

`platform.platform()` и `python_version()` печатают сведения об окружении. Импорт `defaultdict` в дальнейшем не используется. Комментарий про `(100, 1000)` сам настройки не меняет: `BENCH_REQUESTS_PER_CASE` должен оставаться числом; чтобы увеличить выборку, присвой ему, например, `1000`.

**Вопрос преподавателя:** «Почему нельзя заявлять p99 в продакшене по этим данным?» — Это небольшая последовательная выборка на localhost, без распределённой нагрузки, сети и реалистичной конкуренции. Для устойчивой оценки нужны больше данных и повторные запуски.

**Вопрос:** «Почему нужно видеть failures рядом с p95?» — Быстрые успешные ответы не компенсируют большое число провалов; исключение неуспешных запросов из задержек может делать картину чрезмерно благоприятной.


### 10.1 Execute timed requests

The benchmark first creates one snippet for read tests, then sends unique create requests with unique idempotency keys. Each protocol gets its own requests, and the same content is used at each size. The full results are saved to CSV.

In [ ]:
benchmark_rows = []
benchmark_samples = []

# Reuse HTTP connections; avoid creating a new client per request.
with httpx.Client(timeout=BENCH_TIMEOUT_S) as bench_http:
    for size in BENCH_SIZES:
        content = 'x' * size  # ASCII: character count equals UTF-8 byte count
        for protocol in ('gRPC', 'REST'):
            for operation in ('Create', 'Get'):
                times_ms = []
                failures = 0
                req_sizes, resp_sizes = [], []

                if operation == 'Get':
                    # Create a real record before timing read operations.
                    seed = service.create(content, title='benchmark', language='plaintext',
                                          ttl_seconds=3600, key=str(uuid.uuid4()))
                    snippet_id = seed['id']

                for i in range(BENCH_WARMUP + BENCH_REQUESTS_PER_CASE):
                    measured = i >= BENCH_WARMUP
                    try:
                        if operation == 'Create':
                            key = str(uuid.uuid4())
                            payload = {'content': content, 'title': 'benchmark',
                                       'language': 'plaintext', 'ttl_seconds': 3600}
                            if protocol == 'gRPC':
                                req = pb.CreateSnippetRequest(**payload, idempotency_key=key)
                                req_size = len(req.SerializeToString())
                                start = time.perf_counter_ns()
                                result = stub.CreateSnippet(req, timeout=BENCH_TIMEOUT_S)
                                elapsed = (time.perf_counter_ns() - start) / 1e6
                                resp_size = len(result.SerializeToString())
                                assert result.id
                            else:
                                req_size = len(json.dumps(payload, ensure_ascii=False,
                                                          separators=(',', ':')).encode('utf-8'))
                                start = time.perf_counter_ns()
                                response = bench_http.post(f'{BASE_URL}/v1/snippets', json=payload,
                                    headers={'Idempotency-Key': key})
                                elapsed = (time.perf_counter_ns() - start) / 1e6
                                response.raise_for_status()
                                assert response.json()['id']
                                resp_size = len(response.content)
                        else:
                            if protocol == 'gRPC':
                                req = pb.GetSnippetRequest(id=snippet_id)
                                req_size = len(req.SerializeToString())
                                start = time.perf_counter_ns()
                                result = stub.GetSnippet(req, timeout=BENCH_TIMEOUT_S)
                                elapsed = (time.perf_counter_ns() - start) / 1e6
                                assert result.content == content
                                resp_size = len(result.SerializeToString())
                            else:
                                req_size = 0  # GET has no JSON body; URL/headers excluded
                                start = time.perf_counter_ns()
                                response = bench_http.get(f'{BASE_URL}/v1/snippets/{snippet_id}')
                                elapsed = (time.perf_counter_ns() - start) / 1e6
                                response.raise_for_status()
                                assert response.json()['content'] == content
                                resp_size = len(response.content)

                        if measured:
                            times_ms.append(elapsed)
                            req_sizes.append(req_size)
                            resp_sizes.append(resp_size)
                            benchmark_samples.append({'protocol': protocol, 'operation': operation,
                                'content_bytes': size, 'latency_ms': round(elapsed, 4)})
                    except Exception as exc:
                        if measured:
                            failures += 1
                        else:
                            raise RuntimeError(f'Warmup failed: {protocol} {operation} {size}B') from exc

                row = summarize(protocol, operation, size, times_ms, failures,
                    round(statistics.mean(req_sizes), 1) if req_sizes else None,
                    round(statistics.mean(resp_sizes), 1) if resp_sizes else None)
                benchmark_rows.append(row)
                print(f'{protocol:4} {operation:6} {size:5} B | p50={row["p50_ms"]} ms '
                      f'p95={row["p95_ms"]} ms p99={row["p99_ms"]} ms '
                      f'failures={failures}')

assert benchmark_rows, 'No benchmark results'
assert all(row['successes'] > 0 for row in benchmark_rows), 'Some cases have no successful requests'

summary_path = WORKDIR / 'step5_benchmark_summary.csv'
samples_path = WORKDIR / 'step5_benchmark_samples.csv'
for path, rows in ((summary_path, benchmark_rows), (samples_path, benchmark_samples)):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
print('Saved:', summary_path, 'and', samples_path)

### Разбор: выполнение замеров и сохранение CSV

**1. Перебор сценариев.** Три вложенных цикла выбирают размер, протокол и операцию. Строка `'x' * size` состоит из ASCII, поэтому число символов равно размеру UTF-8 в байтах. Для Get реальная запись создаётся заранее через общий сервис, вне измеряемого участка.

**2. Прогрев.** Цикл выполняет `BENCH_WARMUP + BENCH_REQUESTS_PER_CASE` вызовов. Только когда `i >= BENCH_WARMUP`, результат попадает в статистику. Прогрев всё равно создаёт записи при Create и не является «бесплатным», просто его задержки не входят в CSV.

**3. Что измеряется.** `time.perf_counter_ns()` — монотонный счётчик высокой точности, разность делится на миллион для миллисекунд. Время охватывает клиентский сетевой вызов и серверную работу, включая SQLite. Для REST явный разбор `response.json()` выполняется после остановки таймера; у gRPC возврат метода уже содержит разобранный ответ. Подготовка запроса также частично находится вне замера. Поэтому это сравнение данных клиентских путей целиком с их границами, а не чистой скорости сериализаторов.

`stub.CreateSnippet()` / `GetSnippet()` используют существующий канал; один `httpx.Client` переиспользует HTTP-соединения. Это уменьшает влияние создания клиента на каждый запрос. При каждом Create используется новый UUID, чтобы измерять создание, а не возврат результата идемпотентного повтора. Надёжные обёртки здесь **не используются**: бенчмарк делает отдельные вызовы без автоматических повторов.

**4. Размеры.** `SerializeToString()` даёт длину тела Protobuf; для REST Create длина JSON вычисляется отдельно, ответ измеряется через `response.content`. Для REST Get `request_bytes=0`, потому что тела запроса нет, хотя URL и заголовки передаются. Важная асимметрия: UUID включён в тело gRPC Create, но передаётся заголовком REST и исключён из его `request_bytes`. HTTP-заголовки, фрейминг, TCP/TLS здесь не измеряются.

**5. Ошибки.** Проверяются статус/ID/содержимое. Ошибка прогрева прерывает эксперимент, ошибка измеряемого вызова увеличивает `failures` и позволяет продолжить. Типы ошибок при этом не записываются в CSV. Проверка в конце требует хотя бы одного успеха в каждом сценарии, но **не требует нуля ошибок**.

**6. Вывод.** `benchmark_rows` содержит 12 агрегированных строк; `benchmark_samples` — отдельные успешные измерения. `csv.DictWriter` сохраняет [сводку](./sis2_pastebin_runtime/step5_benchmark_summary.csv) и [успешные измерения](./sis2_pastebin_runtime/step5_benchmark_samples.csv). Файлы перезаписываются при новом запуске. `newline=''` нужен для корректной записи CSV без лишних переводов строк.

**Самые важные вызовы:** `perf_counter_ns()`, `SerializeToString()`, `summarize()`, `DictWriter.writerows()`.

**Вопрос преподавателя:** «Порядок gRPC → REST не влияет?» — Может влиять: прогрев, состояние базы и фоновые процессы меняются во времени. Рандомизация порядка и повторные серии сделали бы сравнение надёжнее.

**Вопрос:** «Это тест пропускной способности?» — Нет. Запросы последовательные; здесь сравнивается клиентская задержка при низкой конкуренции. Production RPS из этого не следует.


### 10.2 Compare results and interpret them

A smaller serialized payload does not automatically guarantee lower end-to-end latency. SQLite writes, Python server overhead, and loopback networking can dominate. Read `p95_ms`, `p99_ms`, and failures alongside mean latency.

In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(benchmark_rows).sort_values(['operation', 'content_bytes', 'protocol']))
except ImportError:
    for row in benchmark_rows:
        print(row)

print('\nRelative gRPC vs REST (positive = gRPC improvement):')
for operation in ('Create', 'Get'):
    for size in BENCH_SIZES:
        g = next(r for r in benchmark_rows if r['protocol'] == 'gRPC' and
                 r['operation'] == operation and r['content_bytes'] == size)
        r = next(r for r in benchmark_rows if r['protocol'] == 'REST' and
                 r['operation'] == operation and r['content_bytes'] == size)
        if g['p50_ms'] is not None and r['p50_ms']:
            speedup = 100 * (r['p50_ms'] - g['p50_ms']) / r['p50_ms']
            print(f'{operation:6} {size:5} B: median latency improvement = {speedup:+.1f}% '
                  f'(gRPC {g["p50_ms"]} ms vs REST {r["p50_ms"]} ms)')
        if g['response_bytes'] is not None and r['response_bytes']:
            saved = 100 * (r['response_bytes'] - g['response_bytes']) / r['response_bytes']
            print(f'               response payload reduction = {saved:+.1f}% '
                  f'(gRPC {g["response_bytes"]} B vs REST {r["response_bytes"]} B)')

### Разбор: таблица и относительное сравнение

Если `pandas` установлен, словари превращаются в DataFrame и сортируются по операции, размеру и протоколу. Без `pandas` выводятся исходные словари; сами измерения не меняются. Jupyter-функция `display()` показывает таблицу.

Для каждой пары операции и размера `next(...)` находит строку gRPC и строку REST. Далее вычисляются два разных показателя:

- Снижение медианной задержки: `100 × (REST p50 − gRPC p50) / REST p50`.
- Уменьшение тела ответа: `100 × (REST bytes − gRPC bytes) / REST bytes`.

Положительный процент означает меньшее значение у gRPC, отрицательный — преимущество REST по этому показателю. Например, 1 мс против 2 мс — снижение задержки на 50%; фраза «в два раза быстрее» выражает другое отношение и требует аккуратного пояснения. Метрика тела ответа не описывает весь сетевой трафик.

**Важно:** блок сравнивает `benchmark_rows` в памяти текущего запуска. Если CSV заменён отдельно, вывод может отличаться от него. Новая ячейка графиков ниже намеренно читает сам сохранённый CSV.

**Главные операции:** поиск пар через `next()`, расчёт относительной разницы и представление таблицы. Этот блок интерпретирует результаты, не выполняя новых сетевых вызовов.

**Вопрос преподавателя:** «Почему уменьшение размера ответа заметнее на коротком тексте?» — Фиксированные имена JSON-полей и оформление занимают большую долю короткого ответа; при длинном тексте доминирует само содержимое. Это проверяется числами конкретного запуска.

**Вопрос:** «Можно ли назвать весь gRPC быстрее на процент из строки Create?» — Нет: процент относится к конкретной операции, размеру, медиане и текущему эксперименту.


### 10.3 Визуальный отчёт из сохранённого CSV

Сравнение задержек и тел сообщений: PNG для просмотра и двухстраничный PDF для отчёта. Ячейка работает независимо от серверов; при необходимости установи Matplotlib через `%pip install matplotlib`.


In [ ]:
from pathlib import Path
import csv
import math
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

def build_benchmark_report(summary_csv):
    """Build figures from saved measurements without running servers or benchmarks."""
    summary_csv = Path(summary_csv)
    with summary_csv.open(encoding='utf-8', newline='') as source:
        rows = list(csv.DictReader(source))
    if not rows:
        raise ValueError('The summary CSV is empty; run section 10.1 first.')
    sizes = sorted({int(row['content_bytes']) for row in rows})
    protocols = ('gRPC', 'REST')
    operations = ('Create', 'Get')
    lookup = {(row['operation'], row['protocol'], int(row['content_bytes'])): row for row in rows}
    expected = {(op, proto, size) for op in operations for proto in protocols for size in sizes}
    if len(lookup) != len(rows) or set(lookup) != expected:
        raise ValueError('Expected one row for each protocol / operation / size.')

    def numeric(row, field):
        value = row.get(field)
        if value in (None, '', 'None'):
            return float('nan')
        return float(value)

    attempts = sum(int(row['attempts']) for row in rows)
    successes = sum(int(row['successes']) for row in rows)
    failures = sum(int(row['failures']) for row in rows)
    caption = (f'{len(rows)} cases | {attempts:,} measured attempts | '
               f'{successes:,} successes | {failures:,} failures\n'
               'Sequential localhost / SQLite run. Latencies describe successful calls only; '
               'not a production capacity estimate.')
    colors = {'gRPC': '#237B83', 'REST': '#DB7934'}
    labels = [f'{size:,} B' for size in sizes]
    positions = list(range(len(sizes)))
    width = 0.36

    def bars(ax, operation, metric, ylabel, title, logarithmic=False):
        for index, protocol in enumerate(protocols):
            values = [numeric(lookup[(operation, protocol, size)], metric) for size in sizes]
            offsets = [position + (index - 0.5) * width for position in positions]
            plotted = [value if math.isfinite(value) and (not logarithmic or value > 0)
                       else float('nan') for value in values]
            rectangles = ax.bar(offsets, plotted, width, label=protocol, color=colors[protocol])
            value_labels = [('N/A' if not math.isfinite(value) else
                             f'{value:,.0f}' if metric.endswith('bytes') else f'{value:.3f}')
                            for value in values]
            ax.bar_label(rectangles, labels=value_labels, padding=3, fontsize=8)
        ax.set_xticks(positions, labels)
        ax.set(xlabel='Snippet content size', ylabel=ylabel, title=title)
        if logarithmic:
            ax.set_yscale('log')
            ax.margins(y=0.28)
        else:
            ax.set_ylim(bottom=0)
            ax.margins(y=0.22)
        ax.grid(axis='y', alpha=0.2)
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(frameon=False, fontsize=8)

    with plt.rc_context({'font.family': 'DejaVu Sans', 'font.size': 10}):
        latency_figure, axes = plt.subplots(3, 2, figsize=(13, 12))
        for row_index, (metric, name) in enumerate((('p50_ms', 'p50 / median'),
                                                  ('p95_ms', 'p95'), ('p99_ms', 'p99'))):
            for column, operation in enumerate(operations):
                bars(axes[row_index, column], operation, metric, 'Latency (ms) - lower is better',
                     f'{operation} | {name}')
            # Use a common scale within each percentile row for the two operations.
            upper = max(ax.get_ylim()[1] for ax in axes[row_index])
            for ax in axes[row_index]:
                ax.set_ylim(0, upper)
        latency_figure.suptitle('Pastebin | gRPC vs REST: measured client latency',
                               fontsize=17, fontweight='bold', y=0.99)
        latency_figure.text(0.5, 0.018, caption + '\np99 from 100 samples per case is sensitive to a few slow calls.',
                            ha='center', va='bottom', fontsize=9)
        latency_figure.tight_layout(rect=(0, 0.09, 1, 0.96))

        payload_figure, axes = plt.subplots(2, 2, figsize=(13, 9))
        for row_index, metric in enumerate(('request_bytes', 'response_bytes')):
            for column, operation in enumerate(operations):
                bars(axes[row_index, column], operation, metric, 'Body size (bytes, log scale)',
                     f'{operation} | {metric.replace("_", " ")}', logarithmic=True)
        payload_figure.suptitle('Pastebin | serialized message bodies',
                               fontsize=17, fontweight='bold', y=0.99)
        payload_figure.text(0.5, 0.015,
            'Body measurements only: HTTP headers / URLs and transport framing are excluded.\n'
            'REST GET has no request body (0 B, omitted on log scale); its network request is not empty.\n'
            'Create: the gRPC body includes the idempotency key; the REST header carrying it is excluded.',
            ha='center', va='bottom', fontsize=9)
        payload_figure.tight_layout(rect=(0, 0.11, 1, 0.95))

        output_dir = summary_csv.parent
        latency_path = output_dir / 'step5_latency_overview.png'
        payload_path = output_dir / 'step5_payload_overview.png'
        pdf_path = output_dir / 'step5_benchmark_report.pdf'
        latency_figure.savefig(latency_path, dpi=180)
        payload_figure.savefig(payload_path, dpi=180)
        with PdfPages(pdf_path) as pdf:
            pdf.savefig(latency_figure)
            pdf.savefig(payload_figure)
        plt.show()
        plt.close(latency_figure)
        plt.close(payload_figure)
    for output_path in (latency_path, payload_path, pdf_path):
        print('Saved:', output_path)
    return latency_path, payload_path, pdf_path

report_paths = build_benchmark_report(
    Path.cwd() / 'sis2_pastebin_runtime' / 'step5_benchmark_summary.csv')


### Разбор: читаемый визуальный отчёт по CSV

**Роль блока.** `build_benchmark_report(summary_csv)` читает уже сохранённую [сводку Step 5](./sis2_pastebin_runtime/step5_benchmark_summary.csv) и строит графики. Эту ячейку можно выполнить отдельно: она сама импортирует зависимости и не требует запущенных серверов или `benchmark_rows` в памяти. Нужны существующий CSV, Matplotlib и верная рабочая папка ядра. Если CSV ещё нет, сначала выполни бенчмарк.

**Порядок работы:**

1. `csv.DictReader` читает каждую строку как словарь. CSV хранит текст, поэтому числа преобразуются через `int()` и `float()`. Пустые значения переводятся в `NaN`, чтобы не выдавать отсутствие результата за нулевую задержку.
2. `lookup` сопоставляет ключ `(операция, протокол, размер)` с одной строкой. Проверка `expected` запрещает дубликаты и отсутствующие комбинации в пределах найденных размеров. Итоговые числа попыток, успехов и провалов суммируются для подписи.
3. Вложенная `bars()` рисует парные столбцы: бирюзовый — gRPC, оранжевый — REST. Над ними указаны реальные значения CSV; ниже для задержки означает быстрее. Столбцы по оси X расположены как категории, а не как равномерная числовая шкала размеров.
4. Первая фигура содержит шесть панелей: p50, p95 и p99 отдельно для Create и Get. Внутри одной строки у обеих операций одинаковая шкала Y — это облегчает сравнение. Между строками шкалы могут различаться: ориентируйся на подписи, а не только на высоту.
5. Вторая фигура сравнивает тела запросов и ответов. Логарифмическая ось Y помогает видеть диапазон от десятков до тысяч байт; равные расстояния означают одинаковое отношение, а не одинаковую прибавку. Нулевое тело REST Get не рисуется на логарифмической оси, что явно отмечено под рисунком.
6. `savefig()` сохраняет PNG, а `PdfPages` объединяет обе фигуры в двухстраничный PDF. `plt.show()` показывает их в ноутбуке, `plt.close()` освобождает фигуры после вывода.

**Готовые файлы:** [задержки](./sis2_pastebin_runtime/step5_latency_overview.png), [размеры сообщений](./sis2_pastebin_runtime/step5_payload_overview.png), [PDF для отчёта](./sis2_pastebin_runtime/step5_benchmark_report.pdf). При замене CSV перезапусти только эту ячейку, чтобы обновить визуализацию. Старые PNG — снимки предыдущих данных и сами не обновляются.

**Главный метод:** `build_benchmark_report()`. Он не вычисляет новые p95 из исходных измерений: отображает агрегаты из CSV. Подписи показывают ограничения: последовательный localhost, задержки успешных вызовов, исключение сетевой обвязки из размеров и чувствительность p99 при небольшой выборке.

**Вопрос преподавателя:** «Почему нельзя построить честный boxplot из summary CSV?» — Там нет всех наблюдений или квартилей. Для распределения нужен samples CSV; нельзя восстанавливать его по среднему и трём перцентилям.

**Вопрос:** «Если один столбец ниже, доказано ли превосходство протокола вообще?» — Нет: это значение одного запуска. Доверительные интервалы и разброс между повторными сериями здесь не оценивались.

**Сохранённый визуальный отчёт по текущему CSV:**

![Сравнение p50, p95 и p99 для создания и чтения](./sis2_pastebin_runtime/step5_latency_overview.png)

![Сравнение размеров тел запросов и ответов](./sis2_pastebin_runtime/step5_payload_overview.png)

Для новой серии измерений обнови графики запуском ячейки выше. Значения на этих изображениях относятся к CSV, из которого они построены.


## Step 5 result

After a successful `Run All`, inspect `sis2_pastebin_runtime/step5_benchmark_summary.csv` and the charts. Do not copy example values into your submission: report the measurements produced on your machine. The next cell shuts down the servers.

## 11. Shut down servers

Выполни эту ячейку для остановки серверов текущего ядра. Она проверяет наличие объектов, поэтому работает и после неполного запуска, когда `channel` или серверы ещё не созданы. Если ожидание остановки превышено, появится явная ошибка.

`Run All` остановит серверы здесь после бенчмарков. Для нового запуска перезапусти ядро и выполни ячейки сверху вниз; для просмотра API оставь последнюю ячейку невыполненной. Не запускай Step 3 и Step 5 одновременно: они используют одинаковые порты.

Эта ячейка управляет только объектами текущего ядра. Сервер в отдельном терминале останавливается через Ctrl+C в том терминале. Перезапуск ядра останавливает фоновые серверы этого ядра и сбрасывает его переменные.


In [ ]:
active_channel = globals().get('channel')
if active_channel is not None:
    active_channel.close()

active_grpc_server = globals().get('grpc_server')
if active_grpc_server is not None:
    if not active_grpc_server.stop(grace=1).wait(timeout=3):
        raise RuntimeError('gRPC server did not stop within 3 seconds')

active_rest_server = globals().get('rest_server')
if active_rest_server is not None:
    active_rest_server.should_exit = True

active_rest_thread = globals().get('rest_thread')
if active_rest_thread is not None:
    active_rest_thread.join(timeout=5)
    if active_rest_thread.is_alive():
        raise RuntimeError('REST server did not stop within 5 seconds')

print('Servers stopped (or were not started)')


### Разбор: безопасная остановка ресурсов

**Текущее состояние:** все команды этой ячейки начинаются с `#`. Это комментарии, поэтому сейчас она ничего не останавливает. Чтобы включить остановку, убери начальные `#`, сохранив отступы. Такое состояние оставлено для ручной работы с API после запуска ноутбука.

**Логика при включённом коде.** `globals().get('channel')` ищет объект в пространстве имён ядра. Если его нет, возвращает `None` вместо `NameError`. Найденный канал закрывается через `close()`. Это не выключает сервер, только клиентское соединение.

`grpc_server.stop(grace=1)` прекращает приём новых RPC и даёт текущим секунду завершиться. `.wait(timeout=3)` подтверждает остановку; при превышении ожидания возникает явная ошибка. Для Uvicorn флаг `should_exit=True` сигнализирует выход, `join(timeout=5)` ожидает поток, а `is_alive()` проверяет, что он действительно завершился.

Объекты проверяются перед использованием, поэтому предусмотрен неполный запуск, когда клиентские примеры ещё не выполнялись. При этом ошибка остановки gRPC прервёт ячейку до последующих действий — при такой ошибке перезапуск ядра остаётся способом завершить ресурсы этого ядра.

**Важные методы:** `close()`, `stop()`, `wait()`, `join()`, `is_alive()`. Они управляют жизненным циклом, но не удаляют базу или сохранённые отчёты.

**Вопрос преподавателя:** «Закрытие вкладки `/docs` остановит сервер?» — Нет: он работает в Python-процессе, а браузер — клиент. Нужно выполнить остановку или завершить соответствующее ядро/процесс.

**Вопрос:** «Почему нельзя остановить внешний сервер через переменную ноутбука?» — Переменная указывает на объект только текущего процесса; отдельный терминальный процесс имеет собственную память и завершается отдельно.


## Подготовка к защите: что нужно уметь объяснить

Это ориентиры по коду, а не гарантированный список вопросов преподавателя. Основная история проекта: **один сервис хранения → два транспорта → корректность повторов → измеримое сравнение**.

| Приоритет | Метод или часть | Что показать и объяснить |
|---|---|---|
| 1 | `PastebinService.create()` | Валидация UTF-8-размера и TTL, отпечаток запроса, поиск ключа, транзакция, уникальный ID, фиксация записи и ответа. |
| 1 | `PastebinService.get()` / `cleanup()` | Логическое истечение TTL отделено от физической очистки. |
| 1 | `CreateSnippet` / `GetSnippet` и REST-обработчики | Общая логика, разные форматы сообщений и статусы, нюансы значений по умолчанию. |
| 1 | `call_with_retries()` | Какие ошибки повторяются, общий бюджет, остаток времени, backoff/jitter, ограничения HTTP-дедлайна. |
| 1 | `create_*_reliable()` | Один ключ и неизменное тело на все попытки; таймаут не доказывает, что сервер не создал запись. |
| 1 | `percentile()` / `summarize()` | p50/p95/p99, доля ошибок и исключение неуспешных вызовов из статистики задержек. |
| 1 | Цикл бенчмарка | Границы таймера, прогрев, новые ключи для Create, повторное использование соединений и последовательный порядок. |
| 2 | `build_benchmark_report()` | Как CSV превращается в читаемые фигуры, почему тело сообщения не равно сетевому трафику. |
| 2 | Остановка серверов | Почему заняты порты, чем отличается сервер от клиента и что делает перезапуск ядра. |

**Возможные вопросы по заданию:**

- **«Почему Python, хотя требуется Java?»** — Это Python-прототип для проверки контракта, общей семантики и сравнительного эксперимента. Отклонение нужно прямо указать; эта реализация сама по себе не выполняет требование о языке Java.
- **«Почему добавлен gRPC, если раньше выбран REST?»** — REST служит базовым вариантом, а второй транспорт позволяет измерить различия на одинаковых операциях и общей базе. Из одного локального опыта не следует, что надо менять исходный архитектурный выбор.
- **«Что гарантирует идемпотентность?»** — Для одинакового ключа и параметров возвращается сохранённый ответ без повторного создания; это не обещание, что сеть выполнит доставку ровно один раз.
- **«Почему нельзя объявить победителя по одному графику?»** — Есть зависимость от SQLite, клиента, Python, размера, операции, порядка запусков и шумов. Нужны повторные серии и отдельная нагрузочная методика.
- **«Какие улучшения нужны дальше?»** — Конкурентные проверки ключей, граничные TTL/размеры и истечение срока, уточнение повторяемых ошибок, строгий HTTP-бюджет, явное закрытие SQLite-соединений, очистка ключей, повторные серии и измерение полного трафика. В этом ноутбуке эти улучшения не реализованы автоматически.

Для демонстрации: создай текст через `/docs`, открой полученный URL, покажи повтор с одним ключом и конфликт 409, затем объясни одну пару столбцов p95 и ограничения этих измерений. Не обещай гарантии RPS/p99 для production на основе localhost.


## 12. Полный гайд: как пользоваться Pastebin

### Что мы создали и что открывается в браузере

Да, работающий сервер — результат этой программы. Это локальный сервис Pastebin: он принимает текст, сохраняет его и выдаёт ссылку для чтения. Реализация находится в ноутбуке; во время выполнения Python запускает серверы в фоновых потоках.

Страница [http://127.0.0.1:18080/docs](http://127.0.0.1:18080/docs) — Swagger UI, интерактивная документация REST API. Через неё можно отправлять настоящие запросы к нашему серверу: создавать и читать тексты. Отдельного интерфейса с лентой записей, кнопкой «Мои тексты» и редактором в этом прототипе нет.

| Часть программы | Для чего нужна |
|---|---|
| REST, порт `18080` | HTTP API, документация `/docs` и ссылки на тексты, которые открываются в браузере. |
| gRPC, порт `50071` | Второй способ создавать и читать те же записи из программного клиента. В Safari как сайт не открывается. |
| SQLite | Общая база для обоих серверов. |
| Тесты и бенчмарки | Проверяют корректность и сравнивают протоколы. Их выполнение не требуется для ручного создания текста через `/docs`. |

Путь запроса выглядит так: **браузер → REST/FastAPI → общий сервис → SQLite → ответ браузеру**. В gRPC используется тот же общий сервис, поэтому запись, созданная через gRPC, тоже доступна по HTTP-ссылке.

### 1. Запустить сервер для ручной работы

1. Открой этот ноутбук в VS Code или Jupyter и выбери Python-ядро с установленными зависимостями.
2. Если предыдущие запуски завершились ошибками, сначала перезапусти ядро (**Restart Kernel**). Не запускай одновременно Step 3 и Step 5: у них одинаковые порты.
3. Проверь зависимости из раздела 1. В текущем ноутбуке команда `%pip install ...` закомментирована; если пакеты уже установлены, выполнять её не нужно. Если их нет, убери начальный `#`, выполни установку и при необходимости перезапусти ядро.
4. Выполни код разделов **2 → 3 → 4** по порядку: конфигурация, база, REST-сервер. Для браузера этого достаточно. Для работы с gRPC выполни также раздел 5; для примеров с готовыми ID — раздел 6.
5. Дождись сообщения `REST listening at http://127.0.0.1:18080` и открой [документацию API](http://127.0.0.1:18080/docs).

Пока пользуешься API, оставляй ядро работающим. Не выполняй остановку серверов. В текущей версии код остановки в разделе 11 закомментирован, поэтому эта ячейка сейчас ничего не выключает. Если вернуть код остановки, `Run All` выполнит и его — после этого страницы перестанут открываться.

Адрес `http://127.0.0.1:18080/` без пути может вернуть `{"detail":"Not Found"}`: обработчик главной страницы не предусмотрен. Для работы открывай именно `/docs`.

### 2. Создать свой текст через браузер

1. На странице `/docs` раскрой зелёный блок **POST /v1/snippets**.
2. Нажми **Try it out**.
3. Поле заголовка **idempotency-key** пока оставь пустым: для обычного создания оно необязательно.
4. В поле **Request body** замени пример следующим JSON:

```json
{
  "content": "Привет! Это мой первый текст.\nА это вторая строка.",
  "title": "Моя первая запись",
  "language": "plaintext",
  "ttl_seconds": 3600
}
```

5. Нажми **Execute** один раз.
6. Ниже, в **Server response**, найди фактический статус **201** и **Response body**. Именно это ответ сервера; блок **Example Value** в описании схемы — только образец.

В JSON используются двойные кавычки. `\n` внутри `content` означает перенос строки в сохраняемом тексте. Если текст содержит двойную кавычку, запиши её как `\"`, а обратный слеш — как `\\`.

| Поле | Значение и ограничения |
|---|---|
| `content` | Обязательный текст: от 1 байта до 1 МиБ, то есть 1 048 576 байт UTF-8. Кириллица и эмодзи могут занимать несколько байтов на символ. |
| `title` | Необязательный заголовок, до 200 символов. По умолчанию пустой. |
| `language` | Необязательная строка длиной 1–32 символа, например `plaintext` или `python`. По умолчанию `plaintext`. Это метаданные; код не исполняется, подсветки на текстовой странице нет. |
| `ttl_seconds` | Срок доступности в секундах: от 60 до 2 592 000 включительно. Если поле не передать, используется 604 800 секунд, то есть 7 дней. |

Примеры TTL: `60` — минута, `3600` — час, `86400` — сутки. Удалять остальные необязательные поля из JSON можно; удалять `content` нельзя.

### 3. Получить ссылку и открыть текст

Успешный ответ создания имеет такую структуру. Значения ниже условные — скопируй реальные из своего **Response body**:

```json
{
  "id": "Ab12Cd34",
  "url": "http://127.0.0.1:18080/s/Ab12Cd34",
  "created_at": "2026-10-31T10:00:00Z",
  "expires_at": "2026-10-31T11:00:00Z"
}
```

- **`id`** — идентификатор записи. Сохрани его для обращения к API.
- **`url`** — готовая ссылка на текст. Скопируй значение без кавычек в адресную строку Safari. На странице будет только сохранённый текст.
- **`created_at`** — время создания, **`expires_at`** — время окончания доступности. Суффикс `Z` обозначает UTC; в Алматы местное время на 5 часов больше.

Заголовок записи не подставляется в ссылку: адрес строится по ID. Ссылка выдаётся сразу при создании, отдельно запрашивать её не нужно.

### 4. Прочитать текст вместе с метаданными

Для чтения текста, заголовка, языка и времени открой:

```text
http://127.0.0.1:18080/v1/snippets/ТВОЙ_ID
```

Либо используй Swagger UI:

1. Раскрой **GET /v1/snippets/{snippet_id}**.
2. Нажми **Try it out**.
3. В `snippet_id` вставь только ID, например `Ab12Cd34`, без URL, кавычек и фигурных скобок.
4. Нажми **Execute**. При успехе статус будет **200**, а `content` в JSON будет содержать твой текст.

Для проверки публичной ссылки прямо в Swagger аналогично используй **GET /s/{snippet_id}** — он вернёт простой текст.

### 5. Повторные запросы и ключ идемпотентности

Без ключа каждое нажатие **Execute** в POST создаёт новую запись, даже если текст совпадает. Это ожидаемое поведение.

Если нужно безопасно повторить запрос после потери ответа, перед первым созданием укажи в поле **idempotency-key** новый UUID. Сгенерировать его можно в отдельной Python-ячейке:

```python
import uuid
print(uuid.uuid4())
```

Скопируй напечатанное значение в заголовок. При повторе сохрани **тот же ключ и все поля запроса**: сервер вернёт прежние ID и ссылку. Если изменить текст, заголовок, язык или TTL с тем же ключом, будет **409 Conflict**. Для другой записи используй новый ключ.

Повтор не продлевает TTL и не восстанавливает истёкшую запись: возвращается первоначальный ответ. Для нового срока доступности создай новую запись с новым ключом или без ключа.

### 6. Где посмотреть все записи и можно ли их редактировать

Списка всех записей в API нет. Сохраняй ссылки или ID из ответов. В разделе 6 ноутбука также печатаются ID и публичная ссылка демонстрационной записи.

При стандартном запуске база находится в [sis2_pastebin_runtime/pastebin.sqlite3](./sis2_pastebin_runtime/pastebin.sqlite3). Её можно открыть в просмотрщике SQLite и посмотреть таблицу `snippets`. Если рабочая папка ядра отличается от папки ноутбука, ориентируйся на путь из сообщения `Storage ready`.

Записи неизменяемые: маршрутов редактирования и ручного удаления здесь нет. Чтобы исправить текст, создай новый и получи новую ссылку. По истечении TTL старая запись недоступна через API. Физическое удаление выполняет `service.cleanup()`; автоматически по расписанию он здесь не вызывается.

Перезапуск ядра не удаляет файл базы. При запуске с той же базой ещё не истёкшие записи снова доступны по прежним ссылкам.

### 7. Можно ли отправить ссылку другому человеку

`127.0.0.1` — адрес **того устройства, на котором открыт браузер**. Сейчас сервер слушает только этот локальный адрес. Ссылки работают на твоём компьютере при запущенном сервере; на телефоне или компьютере друга такая ссылка обратится к их устройству, а не к твоему серверу.

Для доступа с других устройств потребуется отдельно настроить сетевой доступ или размещение сервиса. Текущий запуск ноутбука сам по себе сайт в интернете не публикует. В прототипе нет аккаунтов и проверки прав: любой клиент, имеющий доступ к серверу и знающий ID, может прочитать доступную запись.

### 8. Как остановить и снова запустить серверы

**Для остановки:** в разделе 11 убери начальные `#` у строк кода остановки, сохранив отступы после них, и выполни ячейку. Другой способ — **Restart Kernel**: он остановит серверы этого ядра и сбросит переменные. Закрытие вкладки Safari сервер не выключает.

Сервер, запущенный отдельным процессом в терминале, останови через **Ctrl+C** в том терминале. Ячейка ноутбука управляет только объектами своего ядра.

**Для нового запуска:** перезапусти ядро и выполни разделы 2–4 по порядку; при необходимости добавь раздел 5 для gRPC. Не запускай повторно ячейку создания REST-сервера поверх уже работающей: она заново создаёт переменные и может потерять ссылку на прежний сервер. Для использования API после `Run All` не выполняй активную ячейку остановки.

### 9. Что делать при ошибках

| Что видишь | Что это означает и как действовать |
|---|---|
| Safari не может подключиться | Нет доступного HTTP-сервера по этому адресу. Проверь запуск раздела 4, порт `18080`, схему `http://` и состояние ядра. Остановка серверов в конце ноутбука отключает доступ. |
| `address already in use` / `REST server did not start` | При сообщении о занятом порте остался другой сервер. Останови прежний запуск; перезапусти нужное ядро или заверши отдельный процесс в его терминале. Запускай один ноутбук. |
| 404 на `/` | Главная страница не реализована. Открой `/docs`. |
| 404 при чтении записи | ID неверный, запись истекла или сервер использует другую базу. Проверь ID и `expires_at`; при истечении создай новую запись. |
| 422 при создании | Проверь корректность JSON, наличие непустого `content`, TTL, длину заголовка/языка и формат UUID, если ключ указан. Подробность находится в `detail` ответа. |
| 413 | Текст больше 1 МиБ в UTF-8. Сократи его. |
| 409 | Один ключ идемпотентности использован с разными параметрами. Для новой записи возьми новый ключ или оставь заголовок пустым. |
| Страница не открывается на порту `50071` | Это gRPC, а не веб-страница. Для Safari используй порт `18080`. |
| `NameError` при запуске примеров | Не выполнены предыдущие ячейки или ядро перезапущено. Выполни необходимые разделы по порядку. |

**Первый самостоятельный сценарий:** запусти разделы 2–4 → открой `/docs` → POST с собственным `content` → получи 201 → скопируй `url` → открой её в Safari → прочитай текст → по окончании останови серверы.
